# Empirical CAPM Testing on the Russian Stock Market (2018–2024)

## Методология
1. **Time-Series OLS** (Sharpe, 1964; Lintner, 1965): для каждой акции оцениваем α и β
2. **Fama-MacBeth Two-Pass** (1973): проверяем, оценивает ли рынок бета-риск
3. **GRS Test** (Gibbons, Ross, Shanken, 1989): совместная значимость всех альф

**CAPM предсказывает:**
- $\alpha_i = 0$ для всех активов
- $\gamma_0 = 0$ (нулевая бета = безрисковая ставка)
- $\gamma_1 = E[R_m - r_f]$ (бета полностью объясняет кросс-секцию)

**Данные:** Yahoo Finance (.ME), ОФЗ 1Y как $r_f$, рыночный индекс IMOEX.ME

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. ИМПОРТЫ И НАСТРОЙКИ
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
})

print('Все библиотеки загружены.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. КОНФИГУРАЦИЯ
# ─────────────────────────────────────────────────────────────────────────────
START = '2018-01-01'
END   = '2024-12-31'

# Топ-30 ликвидных бумаг ММВБ (суффикс .ME = Московская биржа на Yahoo Finance)
TICKERS = [
    'SBER.ME',   # Сбербанк
    'GAZP.ME',   # Газпром
    'LKOH.ME',   # Лукойл
    'NVTK.ME',   # Новатэк
    'ROSN.ME',   # Роснефть
    'GMKN.ME',   # Норникель
    'TATN.ME',   # Татнефть
    'CHMF.ME',   # Северсталь
    'NLMK.ME',   # НЛМК
    'ALRS.ME',   # Алроса
    'MTSS.ME',   # МТС
    'VTBR.ME',   # ВТБ
    'PIKK.ME',   # ПИК
    'AFLT.ME',   # Аэрофлот
    'RUAL.ME',   # Русал
    'MOEX.ME',   # Московская биржа
    'PLZL.ME',   # Полюс Золото
    'HYDR.ME',   # РусГидро
    'IRAO.ME',   # Интер РАО
    'MAGN.ME',   # ММК
    'SNGS.ME',   # Сургутнефтегаз
    'PHOR.ME',   # ФосАгро
    'MGNT.ME',   # Магнит
    'SBERP.ME',  # Сбербанк преф.
    'TRNFP.ME',  # Транснефть преф.
    'FEES.ME',   # ФСК ЕЭС
    'YNDX.ME',   # Яндекс (торговался до реструктуризации)
    'TCSG.ME',   # TCS Group (Тинькофф)
    'POSI.ME',   # Positive Technologies
    'OZON.ME',   # Ozon
]

MARKET_TICKER = 'IMOEX.ME'   # Индекс Московской биржи
MIN_OBS_FRAC  = 0.60         # Минимум 60% наблюдений для включения акции

print(f'Акций в выборке: {len(TICKERS)}')
print(f'Период: {START} — {END}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. БЕЗРИСКОВАЯ СТАВКА — ОФЗ 1Y (средняя доходность за год)
# ─────────────────────────────────────────────────────────────────────────────
# Источник: данные ЦБ РФ / Московской биржи (средняя доходность ОФЗ ~1 год)
# https://www.cbr.ru/hd_base/zcyc_params/zcyc/
OFZ_ANNUAL = {
    2018: 0.0730,   # средняя ~7.30%
    2019: 0.0680,   # снижение ключевой ставки
    2020: 0.0520,   # COVID: снижение до 4.25%
    2021: 0.0625,   # начало цикла повышения
    2022: 0.1050,   # санкции: скачок до 20%, затем снижение
    2023: 0.0975,   # ~10% среднегодовая
    2024: 0.1520,   # повышение до 21%
}

# Непрерывно начисляемая месячная ставка: r_monthly = ln(1 + r_annual) / 12
def annual_to_monthly_log(r_annual: float) -> float:
    return np.log(1 + r_annual) / 12

# Строим месячный ряд (начало каждого месяца)
date_range = pd.date_range(start=START, end=END, freq='MS')
rf_series  = pd.Series(
    {d: annual_to_monthly_log(OFZ_ANNUAL[d.year]) for d in date_range},
    name='rf'
)
rf_series.index = pd.PeriodIndex(rf_series.index, freq='M')

print('Безрисковая ставка ОФЗ (месячная, логарифмическая):')
rf_annual_check = rf_series.groupby(rf_series.index.year).mean() * 12 * 100
rf_annual_check.name = 'rf % (annualized)'
print(rf_annual_check.apply(lambda x: f'{x:.2f}%').to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. ЗАГРУЗКА ДАННЫХ С YAHOO FINANCE
# ─────────────────────────────────────────────────────────────────────────────
all_tickers = TICKERS + [MARKET_TICKER]

print('Загружаем котировки (monthly, adjusted)...')
raw = yf.download(
    all_tickers,
    start=START,
    end=END,
    interval='1mo',
    auto_adjust=True,
    progress=True,
    group_by='ticker',
)

# Извлекаем цены закрытия
if isinstance(raw.columns, pd.MultiIndex):
    close = raw.xs('Close', axis=1, level=1)
else:
    close = raw[['Close']]

# Приводим индекс к PeriodIndex(M) для совместимости с rf
close.index = pd.PeriodIndex(close.index, freq='M')

print(f'\nЗагружено: {close.shape[0]} месяцев × {close.shape[1]} инструментов')
print(f'Период: {close.index[0]} — {close.index[-1]}')

# Доля пропущенных значений
miss = close.isnull().mean().sort_values(ascending=False)
print('\nДоля пропусков по инструментам:')
print(miss[miss > 0].apply(lambda x: f'{x*100:.1f}%').to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. ОЧИСТКА, ДОХОДНОСТИ, ФИЛЬТРАЦИЯ
# ─────────────────────────────────────────────────────────────────────────────

# Логарифмические месячные доходности (удобны аддитивностью)
log_ret = np.log(close / close.shift(1)).dropna(how='all')

# Рыночный индекс
market_ret = log_ret[MARKET_TICKER].rename('Rm')
stock_ret  = log_ret[TICKERS]

# Фильтр: не менее MIN_OBS_FRAC от общего числа наблюдений рынка
min_obs      = int(MIN_OBS_FRAC * market_ret.count())
valid_mask   = stock_ret.count() >= min_obs
valid_tickers = stock_ret.columns[valid_mask].tolist()
stock_ret    = stock_ret[valid_tickers]

print(f'Акций прошло фильтр (≥{MIN_OBS_FRAC*100:.0f}% наблюдений): {len(valid_tickers)}')
print(valid_tickers)

# Общий временной индекс
common_idx = (
    market_ret.dropna().index
    .intersection(rf_series.index)
    .intersection(stock_ret.dropna(how='all').index)
)

Rm  = market_ret.loc[common_idx]
rf  = rf_series.loc[common_idx]
R   = stock_ret.loc[common_idx]

# Избыточные доходности
MKT = (Rm - rf).rename('MKT')    # рыночная премия
ER  = R.subtract(rf, axis=0)     # excess returns акций

# Убираем суффикс .ME для читаемости таблиц
clean_name = {t: t.replace('.ME', '') for t in valid_tickers}
ER.rename(columns=clean_name, inplace=True)
R.rename(columns=clean_name,  inplace=True)
names = list(clean_name.values())

print(f'\nИтоговый датасет: {len(common_idx)} месяцев × {len(names)} акций')
print(f'Период: {common_idx[0]} — {common_idx[-1]}')
print(f'\nСредняя рыночная премия (мес.): {MKT.mean()*100:.3f}%')
print(f'Средняя рыночная премия (год):  {MKT.mean()*1200:.2f}%')

## 1. Описательная статистика

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. ОПИСАТЕЛЬНАЯ СТАТИСТИКА
# ─────────────────────────────────────────────────────────────────────────────
desc = pd.DataFrame({
    'Mean % (мес.)':     R.mean() * 100,
    'Excess Mean % (мес.)': ER.mean() * 100,
    'Std % (мес.)':      R.std() * 100,
    'Std % (год)':       R.std() * np.sqrt(12) * 100,
    'Sharpe (год)':      R.mean() / R.std() * np.sqrt(12),
    'Skew':              R.skew(),
    'Kurt':              R.kurtosis(),
    'N':                 R.count().astype(int),
}).round(3)

# Добавляем рынок
mkt_row = pd.DataFrame([{
    'Mean % (мес.)':        Rm.mean() * 100,
    'Excess Mean % (мес.)': MKT.mean() * 100,
    'Std % (мес.)':         Rm.std() * 100,
    'Std % (год)':          Rm.std() * np.sqrt(12) * 100,
    'Sharpe (год)':         MKT.mean() / Rm.std() * np.sqrt(12),
    'Skew':                 Rm.skew(),
    'Kurt':                 Rm.kurtosis(),
    'N':                    int(Rm.count()),
}], index=['IMOEX']).round(3)

desc_full = pd.concat([mkt_row, desc])
print('=== Описательная статистика (месячные логарифмические доходности) ===\n')
print(desc_full.to_string())

In [ ]:
# Тепловая карта корреляций
corr_df = ER.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(
    corr_df, mask=mask, ax=ax,
    cmap='RdYlGn', center=0, vmin=-0.2, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.4, cbar_kws={'label': 'Корреляция'}
)
ax.set_title('Матрица корреляций избыточных доходностей (2018–2024)', fontweight='bold')
plt.tight_layout()
plt.savefig('corr_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Time-Series OLS CAPM

Для каждой акции $i$ оцениваем:
$$r_{i,t} - r_{f,t} = \alpha_i + \beta_i \cdot (r_{m,t} - r_{f,t}) + \varepsilon_{i,t}$$

Стандартные ошибки — **HAC (Newey-West)** для учёта автокорреляции.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. TIME-SERIES CAPM OLS
# ─────────────────────────────────────────────────────────────────────────────
ts_results = []
residuals  = {}   # сохраняем для GRS

for name in names:
    mask = ER[name].notna() & MKT.notna()
    y    = ER[name][mask].values
    X    = sm.add_constant(MKT[mask].values)
    X_df = pd.DataFrame(X, columns=['const', 'MKT'], index=MKT[mask].index)

    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 3})

    # Residuals (для GRS нужен полный индекс)
    res_series = pd.Series(np.nan, index=common_idx)
    res_series[mask] = model.resid
    residuals[name] = res_series

    ts_results.append({
        'Ticker':          name,
        'Alpha % (мес.)':  round(model.params[0] * 100, 4),
        't(α)':            round(model.tvalues[0], 2),
        'Beta':            round(model.params[1], 4),
        't(β)':            round(model.tvalues[1], 2),
        'R²':              round(model.rsquared, 4),
        'Adj. R²':         round(model.rsquared_adj, 4),
        'N':               int(model.nobs),
        'α sig':           '*' if abs(model.tvalues[0]) > 1.96 else '',
    })

ts_df = pd.DataFrame(ts_results).set_index('Ticker').sort_values('Beta')

print('=== Time-Series CAPM: OLS с HAC SE (Newey-West, 3 lags) ===\n')
print(ts_df.to_string())

n_sig = (ts_df['α sig'] == '*').sum()
print(f'\nЗначимых α (|t|>1.96): {n_sig}/{len(ts_df)} = {n_sig/len(ts_df)*100:.1f}%')
print(f'Средняя бета:            {ts_df["Beta"].mean():.3f}')
print(f'Медианная бета:          {ts_df["Beta"].median():.3f}')
print(f'Средний R²:              {ts_df["R²"].mean():.3f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. ВИЗУАЛИЗАЦИЯ: БЕТЫ И АЛЬФЫ
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(17, 7))

sig_mask = ts_df['α sig'] == '*'

# --- BETA ---
ax = axes[0]
colors_b = ['#c0392b' if b > 1 else '#2980b9' for b in ts_df['Beta']]
bars = ax.barh(ts_df.index, ts_df['Beta'], color=colors_b, alpha=0.82, edgecolor='white', height=0.7)
ax.axvline(1.0, color='black', ls='--', lw=1.5, label='β = 1 (рынок)')
ax.axvline(0.0, color='gray',  ls='-',  lw=0.7)
# Добавляем значения
for bar, val in zip(bars, ts_df['Beta']):
    ax.text(val + 0.02 if val >= 0 else val - 0.02,
            bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
ax.set_xlabel('Бета (β)')
ax.set_title('Рыночные беты (CAPM)', fontweight='bold')
ax.legend(fontsize=10)

# --- ALPHA ---
ax = axes[1]
colors_a = ['#e74c3c' if s else '#7f8c8d' for s in sig_mask]
bars_a = ax.barh(ts_df.index, ts_df['Alpha % (мес.)'], color=colors_a, alpha=0.82, edgecolor='white', height=0.7)
ax.axvline(0.0, color='black', ls='--', lw=1.5, label='α = 0 (CAPM)')
for bar, val, s in zip(bars_a, ts_df['Alpha % (мес.)'], sig_mask):
    ax.text(val + 0.01 if val >= 0 else val - 0.01,
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}' + (' *' if s else ''),
            va='center', ha='left' if val >= 0 else 'right', fontsize=8)
ax.set_xlabel('Alpha (% / мес.)')
ax.set_title('Альфы CAPM (красный = значимо при 5%)', fontweight='bold')
ax.legend(fontsize=10)

plt.suptitle('Time-Series CAPM: беты и альфы (2018–2024)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('ts_capm_alpha_beta.png', dpi=150, bbox_inches='tight')
plt.show()